# 03. Ethics in Data Science

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week12/03.Ethics-in-Data-Science/notebooks/01_03.Ethics-in-Data-Science.ipynb)

## Learning Objectives
- Understand the 8 Australian AI Ethics Principles and the Australian Privacy Principles (APPs, Privacy Act 1988).
- Identify ethical risks in Automated Decision-Making (ADM) systems (e.g. proxy discrimination, opacity, lack of recourse).
- Implement Human-in-the-Loop (HITL) triage safeguards for consequential interventions.
- Construct and evaluate an Ethical Impact Assessment (EIA) framework for data science projects.


## 1. Automated Decision-Making (ADM) & Risk Scoring
Modern universities and organisations deploy predictive models to identify students who may need academic assistance or bursary support.
However, automated algorithms must be designed responsibly to prevent punitive, black-box decisions.
Let's model a student retention risk scoring system.

In [ ]:
import pandas as pd
import numpy as np

# Cohort data
cohort = pd.DataFrame({
    'student_id': ['ACU-801', 'ACU-802', 'ACU-803', 'ACU-804', 'ACU-805', 'ACU-806'],
    'lms_login_frequency': [4, 28, 12, 2, 35, 18],
    'assessment_avg': [48.5, 84.0, 62.0, 41.0, 91.5, 68.0],
    'attendance_rate': [0.55, 0.95, 0.78, 0.42, 0.98, 0.82],
    'postcode_socio_decile': [2, 8, 4, 1, 9, 5],
    'bursary_need_aud': [3500, 0, 1500, 4500, 0, 1000]
})

# Calculate transparent academic risk score (0 - 100)
# Weights: Assessment Average (50%), LMS Activity (25%), Attendance (25%)
cohort['academic_risk_score'] = (
    (100 - cohort['assessment_avg']) * 0.50
    + (1.0 - (cohort['lms_login_frequency'] / 40.0).clip(0, 1)) * 100 * 0.25
    + (1.0 - cohort['attendance_rate']) * 100 * 0.25
).round(1)

# Naive automated rule (without human oversight)
cohort['auto_flag'] = np.where(
    cohort['academic_risk_score'] >= 45.0,
    'High Risk - Intervention',
    'Low Risk - Standard'
)

display(cohort[['student_id', 'assessment_avg', 'attendance_rate', 'academic_risk_score', 'auto_flag']])

## 2. Auditing Proxy Discrimination
Under Australian anti-discrimination and privacy legislation, models must not introduce indirect bias via proxy variables (e.g. using postcode as a proxy for socioeconomic or demographic background).
Let's audit whether `postcode_socio_decile` is correlated with the outcome.

In [ ]:
corr = cohort[['postcode_socio_decile', 'academic_risk_score']].corr().iloc[0, 1]
print(f'Correlation between Postcode Decile and Academic Risk: {corr:.3f}')

if abs(corr) > 0.40:
    print('⚠️ [ETHICAL WARNING] Postcode is strongly associated with risk.')
    print('Action: Ensure postcodes are strictly excluded from predictive weights.')
else:
    print('✅ Postcode correlation within safe bounds.')

## 3. Human-in-the-Loop (HITL) Safeguards
Australian AI Ethics Principle 7 (*Contestability & Accountability*) states that automated decisions with significant impacts on individuals must have a human appeal and review process.
We implement a decision triage rule: high-risk students in financial need are escalated to a Student Advisor for empathetic case review rather than automated sanctioning.

In [ ]:
def assign_human_oversight(row):
    if row['academic_risk_score'] >= 50.0 and row['bursary_need_aud'] > 0:
        return 'ESCALATE: Student Advisor Case Review Required'
    elif row['academic_risk_score'] >= 45.0:
        return 'Automated Friendly Nudge (Peer Tutoring Invitation)'
    else:
        return 'No Intervention Needed'

cohort['action_plan'] = cohort.apply(assign_human_oversight, axis=1)
display(cohort[['student_id', 'academic_risk_score', 'bursary_need_aud', 'action_plan']])

## 4. Ethical Impact Assessment (EIA) Rubric
Before launching a data science product into production, teams should complete an Ethical Impact Assessment (EIA) evaluating four key dimensions on a 0–5 scale:
1. **Data Minimisation & Consent (APP 3/5)**
2. **Fairness & Non-Discrimination (Principle 3)**
3. **Algorithmic Transparency & Explainability (Principle 5)**
4. **Human Contestability & Recourse (Principle 7)**

In [ ]:
def run_eia(project_title: str, scores: dict[str, int]):
    total = sum(scores.values())
    max_pts = len(scores) * 5
    pct = (total / max_pts) * 100
    
    print(f'=== Ethical Impact Assessment for: {project_title} ===')
    for dim, score in scores.items():
        status = 'PASS' if score >= 3 else 'FAIL (Immediate Remediation)'
        print(f'  • {dim}: {score}/5 -> {status}')
    print(f'\nTotal Score: {total}/{max_pts} ({pct:.1f}%)')
    
    if pct >= 80 and all(s >= 3 for s in scores.values()):
        print('Result: ✅ APPROVED for Production Deployment')
    else:
        print('Result: ❌ BLOCKED from Deployment (Ethical Safeguards Inadequate)')

# Audit of University Early Alert System
early_alert_scores = {
    'Data Minimisation & Consent (APP 3)': 5,
    'Fairness & Non-Discrimination (Principle 3)': 4,
    'Algorithmic Transparency (Principle 5)': 4,
    'Human Contestability (Principle 7)': 5
}
run_eia('ACU Early Alert Retention System', early_alert_scores)

## 5. Practice Exercises

### Exercise: Evaluating an Automated Job Application Screener
An external vendor pitches an automated resume screening AI to your department.
You conduct an EIA audit and find:
- It was trained on historical data from 2010–2018 with significant gender imbalance (Score: 2/5).
- It uses an uninterpretable deep neural network that cannot explain why a candidate was rejected (Score: 1/5).
- Rejected candidates have no right to appeal to a human recruiter (Score: 1/5).
- Consent was collected during the web submission (Score: 4/5).

Run the EIA function and evaluate the outcome.

In [ ]:
# Exercise: Audit the Vendor AI Tool
vendor_scores = {
    'Data Consent & Notice': 4,
    'Historical Fairness & Gender Parity': 2,
    'Explainability & Transparency': 1,
    'Human Recourse & Contestability': 1
}

# --- Student Solution ---
run_eia('Third-Party Resume Screener v1.0', vendor_scores)